In [1]:
import sys
import os
sys.path.append(os.path.dirname(os.path.abspath(os.getcwd())))

In [2]:
os.chdir(os.path.dirname(os.path.abspath(os.getcwd())))

In [3]:
import torch

from src.mslm.utils.setup_train import build_model
from src.mslm.utils.config_loader import cfg

from src.mslm.inference import MultimodalSignLM

/home/giorgio6846/miniconda3/envs/Sign/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
model_parameters = cfg.model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
version = 110
checkpoint = 1
epoch = 99

model_parameters

{'input_size': 111,
 'output_size': 2048,
 'hidden_size': 1024,
 'nhead': 16,
 'ff_dim': 2816,
 'n_layers': 6,
 'encoder_dropout': 0.45,
 'multihead_dropout': 0.4,
 'pool_dim': 256}

In [5]:
def load_model():
    model = build_model(**model_parameters)    

    model_location = f"../outputs/checkpoints/{version}/{checkpoint}/{epoch}/checkpoint.pth" 
    if not os.path.exists(model_location):
        raise FileNotFoundError(
            f"Model not found {model_location}")

    state_dict = torch.load(model_location)

    model.load_state_dict(state_dict["model_state"])

    return model

In [6]:
model = load_model()
model.to(device)

Model Parameters:  {'A': array([[0, 1, 0, ..., 0, 0, 0],
       [1, 0, 1, ..., 0, 0, 0],
       [0, 1, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 1, 0],
       [0, 0, 0, ..., 1, 0, 1],
       [0, 0, 0, ..., 0, 1, 0]], shape=(111, 111)), 'input_size': 111, 'hidden_size': 1024, 'output_size': 2048, 'nhead': 16, 'ff_dim': 2816, 'n_layers': 6, 'max_seq_length': 20, 'pool_dim': 256, 'encoder_dropout': 0.45, 'multihead_dropout': 0.4}
MHARoPE kwargs {'device': None, 'dtype': None}
dim: 1024 num_heads: 16 dim rope 64
Imitator(
  (stgcn): STGCNBlock(
    (gconv): Conv2d(2, 1536, kernel_size=(1, 1), stride=(1, 1))
    (bn1): BatchNorm2d(1536, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (tcn1): Conv2d(512, 512, kernel_size=(3, 1), stride=(1, 1), padding=(1, 0))
    (tcn2): Conv2d(512, 512, kernel_size=(3, 1), stride=(1, 1), padding=(1, 0))
    (tcn3): Conv2d(512, 512, kernel_size=(3, 1), stride=(1, 1), padding=(1, 0))
    (bn2): BatchNorm2d(1536, eps=1e-05, mom

/home/giorgio6846/miniconda3/envs/Sign/lib/python3.11/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Imitator(
  (stgcn): STGCNBlock(
    (gconv): Conv2d(2, 1536, kernel_size=(1, 1), stride=(1, 1))
    (bn1): BatchNorm2d(1536, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (tcn1): Conv2d(512, 512, kernel_size=(3, 1), stride=(1, 1), padding=(1, 0))
    (tcn2): Conv2d(512, 512, kernel_size=(3, 1), stride=(1, 1), padding=(1, 0))
    (tcn3): Conv2d(512, 512, kernel_size=(3, 1), stride=(1, 1), padding=(1, 0))
    (bn2): BatchNorm2d(1536, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
  )
  (linear_hidden): Sequential(
    (0): Conv2d(1536, 1024, kernel_size=(1, 1), stride=(1, 1))
    (1): ReLU()
    (2): BatchNorm2d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerEncoderLayerRoPE(
        (self_attn): MultiheadAttentionRoPE(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=1024, out_features=1024, bias

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "unsloth/gemma-3n-E2B-it-unsloth-bnb-4bit"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",  # o "fp4"
    bnb_4bit_compute_dtype=torch.bfloat16,  # o torch.float16 si no tienes soporte bf16
)
llama_model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config)
tokenizer = AutoTokenizer.from_pretrained(model_id)
embeddings = llama_model.get_input_embeddings()


# llama_model = AutoModelForCausalLM.from_pretrained(model_id)
# tokenizer = AutoTokenizer.from_pretrained(model_id)
# embeddings = llama_model.get_input_embeddings()

/home/giorgio6846/miniconda3/envs/Sign/lib/python3.11/site-packages/transformers/quantizers/auto.py:222: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)
Loading checkpoint shards: 100%|██████████| 3/3 [00:01<00:00,  2.11it/s]


In [8]:
llama_model.to(device)

Gemma3nForConditionalGeneration(
  (model): Gemma3nModel(
    (vision_tower): TimmWrapperModel(
      (timm_model): MobileNetV5Encoder(
        (conv_stem): ConvNormAct(
          (conv): Conv2dSame(3, 64, kernel_size=(3, 3), stride=(2, 2))
          (bn): RmsNormAct2d(
            (drop): Identity()
            (act): GELU(approximate='tanh')
          )
        )
        (blocks): Sequential(
          (0): Sequential(
            (0): EdgeResidual(
              (conv_exp): Conv2dSame(64, 256, kernel_size=(3, 3), stride=(2, 2), bias=False)
              (bn1): RmsNormAct2d(
                (drop): Identity()
                (act): GELU(approximate='tanh')
              )
              (aa): Identity()
              (se): Identity()
              (conv_pwl): Conv2d(256, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
              (bn2): RmsNormAct2d(
                (drop): Identity()
                (act): Identity()
              )
              (drop_path): Identity()
       

In [9]:
from src.mslm.utils import create_dataloaders, prepare_datasets
from torch.utils.data import Subset

In [11]:
h5_file = "../data/processed/dataset_v6_unsloth.hdf5"
train_ratio = 0.8
key_points = 111
batch_size = 64

In [12]:
tr_ds, val_ds, tr_len, val_len = prepare_datasets(h5_file, train_ratio, key_points)
#get sample dataset
sample_tr = Subset(tr_ds, range(0, 1))
sample_val = Subset(val_ds, range(0, 1))

sample_tr_len = len(sample_tr)
sample_val_len = len(sample_val)

_, test_dataloader = create_dataloaders(
    sample_tr,
    sample_val,
    batch_size=2,  # asegúrate que ≤ len(sample)
    num_workers=10
)


KeyError for dataset1/3200, skipping...
KeyError for dataset1/3201, skipping...
Videos:  4118
Train size:	3295
Validation size:	823


In [13]:
for a in test_dataloader:
    print(a[0])

tensor([[[[0.5158, 0.3110],
          [0.3285, 0.3101],
          [0.2078, 0.5520],
          ...,
          [0.2078, 0.2387],
          [0.2130, 0.2550],
          [0.2078, 0.2696]],

         [[0.5158, 0.3109],
          [0.3284, 0.3101],
          [0.2075, 0.5522],
          ...,
          [0.2075, 0.2403],
          [0.2128, 0.2549],
          [0.2075, 0.2711]],

         [[0.5159, 0.3110],
          [0.3283, 0.3102],
          [0.2073, 0.5508],
          ...,
          [0.2073, 0.2402],
          [0.2125, 0.2565],
          [0.2090, 0.2711]],

         ...,

         [[0.5193, 0.3130],
          [0.3332, 0.3088],
          [0.2120, 0.5484],
          ...,
          [0.2120, 0.2411],
          [0.2156, 0.2560],
          [0.2120, 0.2675]],

         [[0.5186, 0.3129],
          [0.3314, 0.3087],
          [0.2119, 0.5502],
          ...,
          [0.2119, 0.2393],
          [0.2155, 0.2542],
          [0.2119, 0.2674]],

         [[0.5187, 0.3120],
          [0.3313, 0.3087],
    

In [14]:
import torch.nn.functional as F

In [15]:
# llama_model.eval()
# mslm = MultimodalSignLM(llama_model, tokenizer, "cuda")
all_embeddings = llama_model.get_input_embeddings().weight.data
del llama_model
# text = "Enumera los pasos descritos:"

In [ ]:
def embeddings_to_text(embeddings: torch.Tensor, all_embeddings: torch.Tensor, tokenizer) -> str:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
    embeddings = embeddings.to(device)

    embedding_matrix = all_embeddings.to(device)  # [V, D]

    embedding_matrix_norm = F.normalize(embedding_matrix, p=2, dim=1)  # [V, D]
    print(embedding_matrix_norm.shape)

    embeddings_norm = F.normalize(embeddings, p=2, dim=1)  # [T, D]

    similarities = torch.matmul(embeddings_norm, embedding_matrix_norm.T)  # [T, V]

    token_ids = torch.argmax(similarities, dim=1).tolist()
    print(f"Token IDs: {token_ids}")
    return tokenizer.decode(token_ids, skip_special_tokens=True)

In [29]:
import gc
torch.cuda.empty_cache()
gc.collect()

0

In [27]:
del mask_data, data, embeds, sign_embed


In [18]:
true_embeds = None
pred_embeds = None

In [ ]:
with torch.no_grad():
    for data, mask_data, embeds, mask_embds in test_dataloader:
        data = data.to(device).to(torch.float32)
        mask_data = mask_data.to(device)

        print(f"Data shape: {data.shape}, Mask shape: {mask_data.shape}")
        sign_embed, _ = model(data, mask_data)
        sign_embed = sign_embed.to("cuda")
        
        sign_embed = sign_embed.to(dtype=torch.bfloat16)

        sign_embed = sign_embed.to("cuda").to(dtype=torch.float32)
        embeds = embeds.to("cuda").to(dtype=torch.float32)
        print(f"Sign embed shape: {sign_embed.shape}, Embeds shape: {embeds.shape}")
        
        embeds = embeds.to(dtype=all_embeddings.dtype)
        true_embeds = embeds
        pred_embeds = sign_embed
        pred_embeds = pred_embeds[:, :4, :][0].to(dtype=all_embeddings.dtype)
        print(embeddings_to_text(pred_embeds, all_embeddings, tokenizer))
        print(embeddings_to_text(embeds[0], all_embeddings, tokenizer))
        
        del mask_data, data, embeds, sign_embed

Data shape: torch.Size([1, 152, 111, 2]), Mask shape: torch.Size([1, 152])
Sign embed shape: torch.Size([1, 20, 2048]), Embeds shape: torch.Size([1, 4, 2048])
torch.Size([262400, 2048])
Token IDs: [2, 1475, 2985, 13942]
coloresaco
torch.Size([262400, 2048])
Token IDs: [2, 4554, 546, 13216]
caramelo


In [42]:
import torch.nn as nn
F.mse_loss(pred_embeds, true_embeds).mean(dim=-1).item()

/tmp/ipykernel_314393/1388084424.py:2: UserWarning: Using a target size (torch.Size([1, 4, 2048])) that is different to the input size (torch.Size([4, 2048])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  F.mse_loss(pred_embeds, true_embeds).mean(dim=-1).item()


0.69140625

In [44]:
(1 - F.cosine_similarity(pred_embeds, true_embeds, dim=-1)).mean().item()

0.4208984375

In [40]:
pred_embeds.shape

torch.Size([4, 2048])

In [39]:
true_embeds.shape

torch.Size([1, 4, 2048])